In [8]:
import numpy as np
import pandas as pd
from scipy.optimize import root
from scipy.special import kv
import matplotlib.pyplot as plt

# ==========================================================
# FILE INPUT
# ==========================================================
file_linee = "sample_nu_formattato_come_Tab_completa.csv"
file_transizioni = "dati_nuovi.asc"

file_mcmc = "risultati_neutrini_osterbrock_5e10.csv"


# soglia che separa i due regimi
MDOT_SWITCH = 1e-2


# ==========================================================
# COSTANTI FISICHE E PARAMETRI SHAKURA-SUNYAEV
# ==========================================================
c = 2.99792458e10
G = 6.67430e-8
h = 6.62607015e-27
kB = 1.380649e-16
sigma = 5.670374419e-5
Msun = 1.98847e33

eta = 0.083
Nr = 2000
Nnu = 400
nu_min_SS = 1e13
nu_max_SS = 1e17


# ==========================================================
# PARAMETRI BALMER - CASE B
# ==========================================================
eV_to_erg = 1.602176634e-12

hydrogen_ionization_energy = 13.6 * eV_to_erg
hydrogen_ionization_frequency = hydrogen_ionization_energy / h

alpha_B_H = 2.59e-13

alpha_eff_balmer = {
    "Ha": 1.16e-13,
    "Hb": 3.07e-14,
    "Hg": 1.29e-14,
    "Hd": 6.79e-15
}

balmer_wavelengths_angstrom = {
    "Ha": 6563.0,
    "Hb": 4861.0,
    "Hg": 4340.0,
    "Hd": 4102.0
}

balmer_names = set(alpha_eff_balmer.keys())


# ==========================================================
# PARAMETRI ADAF
# ==========================================================
ALFA = 0.3
RMIN = 3.0
RMAX = 1000.0
DELTA = 0.0
C1 = 0.5
C3 = 0.3
TNORM = 1.68e-10
beta_default = 0.5


# ==========================================================
# FUNZIONI SS
# ==========================================================
def costruisci_spettro_disco_SS(
    MBH_solar,
    mdot,
    eta=0.083,
    Nr=2000,
    Nnu=400,
    nu_min=1e13,
    nu_max=1e17
):

    MBH = MBH_solar * Msun

    Ledd = 1.3e38 * MBH_solar

    Ldisk = mdot * Ledd

    Mdot = Ldisk / (eta * c**2)

    Rs = 2 * G * MBH / c**2

    Rin = 3 * Rs
    Rout = 3000 * Rs

    R = np.logspace(
        np.log10(Rin),
        np.log10(Rout),
        Nr
    )

    f = 1.0 - np.sqrt(Rin / R)
    f = np.clip(f, 0.0, None)

    T = (
        (3 * G * MBH * Mdot)
        / (8 * np.pi * sigma * R**3)
        * f
    )**0.25

    nu = np.logspace(
        np.log10(nu_min),
        np.log10(nu_max),
        Nnu
    )

    Bnu = np.zeros((Nnu, Nr))

    for j in range(Nr):

        if T[j] <= 0:
            continue

        x = h * nu / (kB * T[j])

        x = np.clip(
            x,
            1e-12,
            700.0
        )

        Bnu[:, j] = (
            2 * h * nu**3 / c**2
        ) / (
            np.exp(x) - 1.0
        )

    integrand = R[None, :] * Bnu

    I_nu = np.trapezoid(
        integrand,
        R,
        axis=1
    )

    Lnu = 4 * np.pi**2 * I_nu

    return nu, Lnu


# ==========================================================
# FUNZIONI ADAF
# ==========================================================

def bessk(n, x):
    return kv(n, x)


def g(teta):

    return (
        (1.0 / bessk(2, 1.0 / teta))
        * (2.0 + 2.0 * teta + 1.0 / teta)
        * np.exp(-1.0 / teta)
    )


def temperat(y, mdot, m, beta):

    ff = y[2]**2
    xm = 1e3 * y[1]**2
    teta = y[0]**2

    s1 = (
        1.42e9
        * ALFA**(-0.5)
        * np.sqrt(1 - beta)
        * C1**(-0.5)
        * C3**0.5
    )

    s2 = 1.19e-13 * xm

    q1 = (
        1.2e4
        * g(teta)
        * ALFA**(-2)
        * C1**(-2)
        * C3
        * beta
        * m
        * mdot**2
        / RMIN
    )

    q2 = (
        DELTA
        * 9.39e4
        * ((1 - beta) / ff)
        * C3
        * m
        * mdot
        / RMIN
    )

    q = q1 + q2

    nup = (
        s1
        * s2
        * m**(-0.5)
        * mdot**0.5
        * (teta / TNORM)**2
        * RMIN**(-5/4)
    )

    psinc = (
        53.0
        * (xm / 1000.0)**3
        * (teta / (1e9 * TNORM))**7
        * m**0.5
        * mdot**1.5
    )

    abig = (
        1
        + 4 * teta
        + 16 * teta**2
    )

    tau = 23.87 * mdot

    alfac = (
        -np.log(tau)
        / np.log(abig)
    )

    # CORRETTO:
    # il fattore teta deve essere normalizzato con 1e9 * TNORM
    graffa = (
        (
            (
                6.2e7
                * (
                    teta
                    / (1e9 * TNORM)
                )
            )
            / (
                nup
                * 1e-12
            )
        )**(1 - alfac)
        - 1
    )

    pcomp = (
        (psinc / 0.71)
        / (1 - alfac)
        * graffa
    )

    if teta < 1:

        f = (
            4
            * np.sqrt(
                2 * teta / np.pi**3
            )
            * (
                1
                + 1.781 * teta**1.34
            )
        )

    else:

        f = (
            9 * teta
            / (2 * np.pi)
        ) * (
            np.log(
                1.123 * teta + 0.48
            )
            + 1.5
        )

    pbrems = (
        4.78
        * ALFA**(-2)
        * C1**(-2)
        * np.log(RMAX / RMIN)
        * f
        * m
        * mdot**2
    )

    f1 = (
        q
        - psinc
        - pcomp
        - pbrems
    )

    f2 = (
        (1 - ff)
        * 9.39e4
        * ((1 - beta) / ff)
        * C3
        * m
        * mdot
        / RMIN
        - q
    )

    vv = xm**(1/3)

    f3 = (
        vv
        + 1.852 * np.log(vv)
        - 10.36
        - 0.26 * np.log(m * mdot)
        + 0.26
        * np.log(
            teta**3
            * bessk(
                2,
                1 / teta
            )
        )
    )

    return [
        f1,
        f3,
        f2
    ]


def spectrum(
    te,
    f,
    xm,
    m,
    mdot,
    beta
):

    numin = 1e10
    numax = 1e21
    num = 100

    lnu = np.linspace(
        np.log10(numin),
        np.log10(numax),
        num
    )

    nu = 10**lnu

    teta = TNORM * te

    s1 = (
        1.42e9
        * ALFA**(-0.5)
        * np.sqrt(1 - beta)
        * C1**(-0.5)
        * C3**0.5
    )

    s2 = 1.19e-13 * xm

    s3 = 1.05e-24

    abig = (
        1
        + 4 * teta
        + 16 * teta**2
    )

    tau = 23.87 * mdot

    alfac = (
        -np.log(tau)
        / np.log(abig)
    )

    lnup = np.log10(
        s1
        * s2
        * m**(-0.5)
        * mdot**0.5
        * te**2
        * RMIN**(-5/4)
    )

    lnucmax = (
        np.log10(te)
        + 10.796
    )

    llsinc = (
        3 * np.log10(s1 * s2)
        + np.log10(s3)
        - (7/4) * np.log10(RMIN)
        + np.log10(
            m**0.5
            * mdot**1.5
        )
        + 7 * np.log10(te)
    )

    lls = np.zeros(num)
    llc = np.zeros(num)
    llb = np.zeros(num)

    for i in range(num):

        # -----------------------
        # Sincrotrone
        # -----------------------

        if lnu[i] < lnup:

            lls[i] = (
                np.log10(s3)
                + (8/5)
                * np.log10(s1 * s2)
                + np.log10(
                    m**(6/5)
                    * mdot**(4/5)
                )
                + (21/5)
                * np.log10(te)
                + (2/5)
                * lnu[i]
            )

        else:

            lls[i] = -np.inf

        # -----------------------
        # Compton
        # -----------------------

        if lnu[i] > lnup:

            base = (
                llsinc
                + (alfac - 1) * lnup
                - alfac * lnu[i]
            )

            cutoff = (
                -(
                    10**(
                        lnu[i]
                        - lnucmax
                    )
                )
                / np.log(10)
            )

            llc[i] = (
                base
                + cutoff
            )

        else:

            llc[i] = -np.inf

        # -----------------------
        # Bremsstrahlung
        # -----------------------

        if teta < 1:

            fb = (
                4
                * np.sqrt(
                    2 * teta
                    / np.pi**3
                )
                * (
                    1
                    + 1.781
                    * teta**1.34
                )
            )

        else:

            fb = (
                9 * teta
                / (2 * np.pi)
            ) * (
                np.log(
                    1.123 * teta
                    + 0.48
                )
                + 1.5
            )

        base_b = (
            24.36
            - 2
            * np.log10(
                ALFA * C1
            )
            + np.log10(
                np.log(
                    RMAX / RMIN
                )
                * fb
            )
            - np.log10(te)
            + np.log10(
                m
                * mdot**2
            )
        )

        cutoff_b = (
            -(
                4.8e-11
                * nu[i]
                / te
            )
            / np.log(10)
        )

        llb[i] = (
            base_b
            + cutoff_b
        )

    comp_sync = np.where(
        np.isfinite(lls),
        10**lls,
        0.0
    )

    comp_comp = np.where(
        np.isfinite(llc),
        10**llc,
        0.0
    )

    comp_brems = np.where(
        np.isfinite(llb),
        10**llb,
        0.0
    )

    flux = (
        comp_sync
        + comp_comp
        + comp_brems
    )

    log_nuLnu = (
        lnu
        + np.log10(flux)
    )

    return (
        lnu,
        log_nuLnu
    )


def specdata(
    m,
    mdot,
    beta
):

    y0 = [
        0.5,
        1.0,
        0.99
    ]

    sol = root(
        temperat,
        y0,
        args=(
            mdot,
            m,
            beta
        ),
        tol=1e-10
    )

    if not sol.success:

        raise RuntimeError(
            f"Soluzione ADAF non converge per "
            f"m={m}, "
            f"mdot={mdot}, "
            f"beta={beta}"
        )

    y = sol.x

    teta = y[0]**2

    te = (
        teta
        / TNORM
    )

    xm = (
        1e3
        * y[1]**2
    )

    f = (
        y[2]**2
    )

    lnu, log_nuLnu = spectrum(
        te=te,
        f=f,
        xm=xm,
        m=m,
        mdot=mdot,
        beta=beta
    )

    return (
        lnu,
        log_nuLnu
    )


def costruisci_spettro_adaf(
    m,
    mdot,
    beta
):

    lnu, log_nuLnu = specdata(
        m=m,
        mdot=mdot,
        beta=beta
    )

    nu = (
        10**lnu
    )

    nuLnu = (
        10**log_nuLnu
    )

    Lnu = (
        nuLnu
        / nu
    )

    return (
        nu,
        Lnu
    )


# ==========================================================
# FUNZIONI COMUNI DI SUPPORTO
# ==========================================================
def trova_colonna(
    df,
    possibili_nomi,
    obbligatoria=True
):

    for nome in possibili_nomi:

        if nome in df.columns:
            return nome

    if obbligatoria:

        raise ValueError(
            f"Non trovo nessuna di queste colonne: "
            f"{possibili_nomi}"
        )

    return None


def leggi_transizioni(
    file_transizioni
):

    names, Eion, nu_ion_list = np.genfromtxt(
        file_transizioni,
        dtype=(
            "U50",
            float,
            float
        ),
        unpack=True
    )

    return pd.DataFrame({

        "line_name": names,

        "E_ion": Eion,

        "nu_ion": nu_ion_list
    })


def estrai_linee_osservate(
    row,
    norm_columns
):

    risultati = []

    for col in norm_columns:

        valore = row[col]

        if (
            pd.isna(valore)
            or valore <= 0
        ):
            continue

        line_name = (
            col.replace(
                "_norm",
                ""
            )
        )

        # Nel file sample_nu_formattato_come_Tab_completa.csv
        # le luminosita sono gia espresse nel formato richiesto:
        # NON moltiplicare nuovamente per 1e42.
        L_line = valore

        risultati.append({

            "line_name": line_name,

            "L_line": L_line
        })

    return pd.DataFrame(
        risultati
    )


# ==========================================================
# QUANTITA' CHE GUIDANO LE LINEE
# ==========================================================
def calcola_quantita_per_linee(
    nu,
    Lnu,
    df_trans,
    nu_max=None
):

    risultati = []

    if nu_max is None:
        nu_max = np.max(nu)

    # ------------------------------------------------------
    # Q(H), comune a tutte le righe di Balmer
    # ------------------------------------------------------
    mask_H = (
        (nu >= hydrogen_ionization_frequency)
        & (nu <= nu_max)
    )

    if np.sum(mask_H) < 2:

        Q_H = 0.0

    else:

        Q_H = np.trapezoid(
            Lnu[mask_H]
            / (
                h
                * nu[mask_H]
            ),
            nu[mask_H]
        )

    # ------------------------------------------------------
    # Quantita' specifiche di ogni riga
    # ------------------------------------------------------
    for _, row in df_trans.iterrows():

        line_name = row["line_name"]

        # ==================================================
        # BALMER:
        #
        # L_line = Cf * h*nu_line
        #          * (alpha_eff/alpha_B) * Q(H)
        #
        # quindi:
        #
        # Cf = L_line_obs /
        #      [h*nu_line*(alpha_eff/alpha_B)*Q(H)]
        # ==================================================
        if line_name in balmer_names:

            wavelength_A = (
                balmer_wavelengths_angstrom[
                    line_name
                ]
            )

            nu_line = (
                c
                / (
                    wavelength_A
                    * 1e-8
                )
            )

            alpha_eff = (
                alpha_eff_balmer[
                    line_name
                ]
            )

            alpha_ratio = (
                alpha_eff
                / alpha_B_H
            )

            L_line_per_Cf = (
                h
                * nu_line
                * alpha_ratio
                * Q_H
            )

            risultati.append({

                "line_name":
                    line_name,

                "line_model":
                    "Balmer_CaseB",

                "E_ion":
                    13.6,

                "nu_ion":
                    hydrogen_ionization_frequency,

                "L_ion":
                    np.nan,

                "Q_H":
                    Q_H,

                "nu_line":
                    nu_line,

                "alpha_eff":
                    alpha_eff,

                "alpha_B":
                    alpha_B_H,

                "alpha_ratio":
                    alpha_ratio,

                "L_line_per_Cf":
                    L_line_per_Cf
            })

        # ==================================================
        # METALLI:
        #
        # L_line = Cf * L_ion
        #
        # quindi:
        #
        # Cf = L_line_obs / L_ion
        # ==================================================
        else:

            E_ion = row["E_ion"]
            nu_ion = row["nu_ion"]

            if nu_ion >= nu_max:

                L_ion = 0.0

            else:

                mask = (
                    (nu >= nu_ion)
                    & (nu <= nu_max)
                )

                if np.sum(mask) < 2:

                    L_ion = 0.0

                else:

                    L_ion = np.trapezoid(
                        Lnu[mask],
                        nu[mask]
                    )

            risultati.append({

                "line_name":
                    line_name,

                "line_model":
                    "Cf_times_Lion",

                "E_ion":
                    E_ion,

                "nu_ion":
                    nu_ion,

                "L_ion":
                    L_ion,

                "Q_H":
                    np.nan,

                "nu_line":
                    np.nan,

                "alpha_eff":
                    np.nan,

                "alpha_B":
                    np.nan,

                "alpha_ratio":
                    np.nan,

                "L_line_per_Cf":
                    L_ion
            })

    return pd.DataFrame(
        risultati
    )


# ==========================================================
# FUNZIONE DOPPIO RAMO
# ==========================================================
def analizza_blazar_doppio_ramo(
    row,
    df_trans,
    norm_columns
):

    nome_blazar = row["label"]

    m = row["massa"]

    mdot = row["mdot"]

    # ======================================================
    # SCELTA DEL REGIME
    # ======================================================
    if mdot < MDOT_SWITCH:

        regime = "ADAF"

        try:

            nu, Lnu = costruisci_spettro_adaf(
                m=m,
                mdot=mdot,
                beta=beta_default
            )

        except RuntimeError as e:

            print(
                f"Attenzione: salto "
                f"{nome_blazar}. "
                f"Motivo: {e}"
            )

            return pd.DataFrame()

        nu_max_model = np.max(nu)

    else:

        regime = "SS"

        nu, Lnu = costruisci_spettro_disco_SS(
            MBH_solar=m,
            mdot=mdot,
            eta=eta,
            Nr=Nr,
            Nnu=Nnu,
            nu_min=nu_min_SS,
            nu_max=nu_max_SS
        )

        nu_max_model = nu_max_SS

    # ======================================================
    # QUANTITA' TEORICHE PER LE LINEE
    # ======================================================
    df_model = calcola_quantita_per_linee(
        nu=nu,
        Lnu=Lnu,
        df_trans=df_trans,
        nu_max=nu_max_model
    )

    # ======================================================
    # LINEE OSSERVATE
    # ======================================================
    df_linee = estrai_linee_osservate(
        row=row,
        norm_columns=norm_columns
    )

    if df_linee.empty:

        return pd.DataFrame()

    # ======================================================
    # ASSOCIO OSSERVAZIONI E MODELLO
    # ======================================================
    df_conf = df_linee.merge(
        df_model,
        on="line_name",
        how="left"
    )

    # ======================================================
    # COVERING FACTOR
    #
    # Per Balmer:
    # Cf = Lobs /
    #      [h nu_line (alpha_eff/alpha_B) Q(H)]
    #
    # Per MgII, CIII, CIV:
    # Cf = Lobs / L_ion
    # ======================================================
    df_conf["covering_factor"] = (
        df_conf["L_line"]
        / df_conf["L_line_per_Cf"]
    )

    df_conf.loc[
        (
            df_conf["L_line_per_Cf"].isna()
        )
        |
        (
            df_conf["L_line_per_Cf"] <= 0
        ),
        "covering_factor"
    ] = np.nan

    # ======================================================
    # INFORMAZIONI SUL BLAZAR
    # ======================================================
    df_conf["label"] = nome_blazar

    df_conf["regime"] = regime

    df_conf["massa"] = m

    df_conf["mdot"] = mdot

    return df_conf[[

        "label",

        "regime",

        "massa",

        "mdot",

        "line_name",

        "line_model",

        "L_line",

        "L_ion",

        "Q_H",

        "nu_line",

        "alpha_eff",

        "alpha_B",

        "alpha_ratio",

        "L_line_per_Cf",

        "covering_factor"

    ]]


# ==========================================================
# LETTURA FILE OSSERVAZIONI E TRANSIZIONI
# ==========================================================
df_tab = pd.read_csv(
    file_linee
)

df_trans = leggi_transizioni(
    file_transizioni
)


col_name_tab = trova_colonna(
    df_tab,
    [
        "label",
        "file_name",
        "name",
        "nome",
        "blazar"
    ]
)

df_tab = df_tab.rename(
    columns={
        col_name_tab:
        "label_originale"
    }
).copy()


df_tab["label_originale"] = (
    df_tab["label_originale"]
    .astype(str)
    .str.strip()
)


# Manteniamo il nome completo del blazar per il merge
df_tab["label"] = df_tab["label_originale"]


norm_columns = [

    col

    for col in df_tab.columns

    if col.endswith("_norm")

]


# ==========================================================
# FUNZIONE CHE ESEGUE IL CROSS-CHECK PER UNA COPPIA MASSA/MDOT
# ==========================================================
def esegui_crosscheck(df_mcmc, col_massa, col_mdot, etichetta):

    print()
    print("=" * 100)
    print(f"CROSS-CHECK COVERING FACTOR: {etichetta}")
    print(f"Uso {col_massa} e {col_mdot}")
    print("=" * 100)

    df_param = df_mcmc[[
        "nome_blazar",
        col_massa,
        col_mdot
    ]].copy()

    df_param = df_param.rename(columns={
        "nome_blazar": "label",
        col_massa: "massa",
        col_mdot: "mdot"
    })

    df_param["label"] = (
        df_param["label"]
        .astype(str)
        .str.strip()
    )

    # Merge sul NOME COMPLETO del blazar.
    df_all = df_tab.merge(
        df_param,
        on="label",
        how="inner"
    )

    if len(df_all) == 0:
        raise ValueError(
            f"Il merge non ha trovato blazar in comune per {etichetta}. "
            "Controlla che i nomi nei due CSV coincidano."
        )

    print(f"Blazar trovati in comune: {len(df_all)}")

    tutti_risultati = []

    for i in range(len(df_all)):
        row = df_all.iloc[i]

        df_conf = analizza_blazar_doppio_ramo(
            row=row,
            df_trans=df_trans,
            norm_columns=norm_columns
        )

        if not df_conf.empty:
            tutti_risultati.append(df_conf)

    if len(tutti_risultati) == 0:
        raise ValueError(
            f"Nessun covering factor calcolato per {etichetta}."
        )

    df_cf = pd.concat(
        tutti_risultati,
        ignore_index=True
    )

    print()
    print(f"TABELLA COMPLETA LINEA PER LINEA - {etichetta}")
    print(df_cf.to_string(index=False))

    # ------------------------------------------------------
    # TABELLA PIVOT DEI COVERING FACTOR
    # ------------------------------------------------------
    cf_pivot = df_cf.pivot_table(
        index="label",
        columns="line_name",
        values="covering_factor",
        aggfunc="first"
    )

    info = (
        df_cf[[
            "label",
            "regime",
            "massa",
            "mdot"
        ]]
        .drop_duplicates(subset="label")
        .set_index("label")
    )

    cf_pivot["Cf_mean"] = cf_pivot.mean(
        axis=1,
        skipna=True
    )

    tabella_finale = info.join(
        cf_pivot,
        how="inner"
    )

    ordine_linee = [
        "CIII",
        "CIV",
        "MgII",
        "HeII",
        "Ha",
        "Hb",
        "Hg",
        "Hd"
    ]

    colonne_presenti = [
        c for c in ordine_linee
        if c in tabella_finale.columns
    ]

    tabella_finale = tabella_finale[[
        "regime",
        "massa",
        "mdot"
    ] + colonne_presenti + [
        "Cf_mean"
    ]].reset_index()

    print()
    print(f"TABELLA FINALE COVERING FACTOR - {etichetta}")
    print(tabella_finale.to_string(index=False))

    return df_cf, tabella_finale


# ==========================================================
# LETTURA PARAMETRI OSTERBROCK 10% E 5%
# ==========================================================
#df_mcmc = pd.read_csv(file_mcmc)
df_mcmc = pd.read_csv(
    file_mcmc,
    sep=r"\s{2,}",
    engine="python"
)

# Pulizia minima dei nomi delle colonne, utile anche se il CSV
# proviene da un file tabulato o contiene spazi accidentali.
df_mcmc.columns = [str(c).strip() for c in df_mcmc.columns]

colonne_richieste = [
    "nome_blazar",
    "massa_10",
    "mdot_10",
    "massa_5",
    "mdot_5"
]

mancanti = [c for c in colonne_richieste if c not in df_mcmc.columns]
if mancanti:
    raise ValueError(
        f"Nel file {file_mcmc} mancano le colonne: {mancanti}"
    )


# ==========================================================
# 1) ESECUZIONE CON ERRORE 10%
# ==========================================================
df_cf_10, tabella_finale_10 = esegui_crosscheck(
    df_mcmc=df_mcmc,
    col_massa="massa_10",
    col_mdot="mdot_10",
    etichetta="OSTERBROCK 10%"
)




# ==========================================================
# 2) ESECUZIONE CON ERRORE 5%
# ==========================================================
df_cf_5, tabella_finale_5 = esegui_crosscheck(
    df_mcmc=df_mcmc,
    col_massa="massa_5",
    col_mdot="mdot_5",
    etichetta="OSTERBROCK 5%"
)





# ==========================================================
# CONFRONTO FINALE 10% vs 5%
# ==========================================================
confronto_10_5 = tabella_finale_10.merge(
    tabella_finale_5,
    on="label",
    how="outer",
    suffixes=("_10", "_5")
)



print()
print("=" * 100)
print("CALCOLI COMPLETATI")
print("Creati i file:")
print("  covering_factor_linee_10percento.csv")
print("  covering_factor_finale_10percento.csv")
print("  covering_factor_linee_5percento.csv")
print("  covering_factor_finale_5percento.csv")
print("  covering_factor_confronto_10_5.csv")
print("=" * 100)



CROSS-CHECK COVERING FACTOR: OSTERBROCK 10%
Uso massa_10 e mdot_10
Blazar trovati in comune: 14

TABELLA COMPLETA LINEA PER LINEA - OSTERBROCK 10%
                    label regime        massa     mdot line_name    line_model       L_line        L_ion          Q_H      nu_line    alpha_eff      alpha_B  alpha_ratio  L_line_per_Cf  covering_factor
    CRATES J024445+132002     SS 1.005000e+09 0.035090      MgII Cf_times_Lion 1.500000e+44 1.576994e+45          NaN          NaN          NaN          NaN          NaN   1.576994e+45         0.095118
             PKS 0446+112   ADAF 4.834000e+10 0.009783       CIV Cf_times_Lion 1.260000e+44 2.542330e+45          NaN          NaN          NaN          NaN          NaN   2.542330e+45         0.049561
             PKS 0446+112   ADAF 4.834000e+10 0.009783      HeII Cf_times_Lion 1.400000e+43 2.528637e+45          NaN          NaN          NaN          NaN          NaN   2.528637e+45         0.005537
      NVSS J054341+062553     SS 3.162000e+0

In [9]:
# Copia del dataframe linea-per-linea
df_plot = df_cf_mcmc.copy()

# Converto Cf in percentuale
df_plot["Cf_percent"] = df_plot["covering_factor"] * 100

# Ordine dei blazar
blazars = df_plot["label"].unique()
x_positions = np.arange(len(blazars))

# Mappa nome blazar -> posizione numerica sull'asse x
x_map = {name: i for i, name in enumerate(blazars)}

# Simboli diversi per le diverse righe
markers = {
    "CIII": "o",
    "CIV": "s",
    "MgII": "^",
    "Ha": "D",
    "Hb": "v",
    "HeII": "P"
}

fig, ax = plt.subplots(figsize=(13, 6))

# Disegno separatamente ciascuna riga
for line_name in df_plot["line_name"].unique():

    subset = df_plot[df_plot["line_name"] == line_name]

    x = [
        x_map[label]
        for label in subset["label"]
    ]

    y = subset["Cf_percent"]

    ax.scatter(
        x,
        y,
        marker=markers.get(line_name, "o"),
        s=55,
        label=line_name
    )

# Linea corrispondente al Cf assunto nell'MCMC
ax.axhline(
    10,
    linestyle="--",
    linewidth=1.5,
    label="Cf = 10%"
)

# Area tra 5 e 20 %
ax.axhspan(
    5,
    20,
    color="lightblue",
    alpha=0.18
)


# Asse x
ax.set_xticks(x_positions)
ax.set_xticklabels(
    blazars,
    rotation=90
)

# Etichette
ax.set_xlabel("Blazar")
ax.set_ylabel("Covering factor (%)")

ax.set_ylim(0.001, 60)
#ax.set_ylim(0.001, 9600)

ax.legend(
    loc="upper right")

plt.tight_layout()
plt.show()

NameError: name 'df_cf_mcmc' is not defined